# Test Suite 03: Gold Layer & Star Schema Validation
**Target:** `gold.fact_listening`, `gold.dim_*`, `gold.ml_*`, `gold.*` marts

**Tests Covered:**
- T1: Fact vs Silver row count parity & PK uniqueness
- T2: Referential integrity & zero orphan foreign keys across all 5 dimensions
- T3: Slowly Changing Dimension (SCD Type 2) `dim_user` integrity
- T4: Metric conservation (`SUM(minutes_played)` matches Silver `SUM(ms_played)/60000`)
- T5: Change Data Feed (CDF) incremental idempotency
- T6: Subscription propensity calibration match rate
- T7: Market share sum completeness (100% per monthly cohort)


In [ ]:
%run ../notebooks/00_config

spark.sql(f"USE CATALOG {CATALOG_NAME}")
print("Testing Gold Star Schema and Data Marts...")


### Test T1: Fact vs Silver Row Count Parity & PK Uniqueness


In [ ]:
fact_cnt = spark.table(GOLD['fact_listening']).count()
silver_cnt = spark.table(SILVER_EVENTS).count()
fact_distinct = spark.table(GOLD['fact_listening']).select("event_id").distinct().count()

assert fact_cnt == silver_cnt, f"FAIL T1: Fact count ({fact_cnt:,}) != Silver count ({silver_cnt:,})"
assert fact_cnt == fact_distinct, f"FAIL T1: Duplicate PKs detected in fact ({fact_cnt - fact_distinct})"

print(f"[PASS] T1: Parity verified: Fact ({fact_cnt:,}) == Silver ({silver_cnt:,}) with 100% unique PKs.")


### Test T2: Referential Integrity & Zero Orphan Foreign Keys


In [ ]:
fact = spark.table(GOLD['fact_listening'])
dim_u = spark.table(GOLD['dim_user'])
dim_t = spark.table(GOLD['dim_track'])
dim_d = spark.table(GOLD['dim_device'])
dim_dt = spark.table(GOLD['dim_date'])
dim_tm = spark.table(GOLD['dim_time'])

orphan_users = fact.join(dim_u, "user_sk", "left_anti").count()
orphan_tracks = fact.join(dim_t, "track_sk", "left_anti").count()
orphan_devices = fact.join(dim_d, "device_sk", "left_anti").count()
orphan_dates = fact.join(dim_dt, "date_key", "left_anti").count()
orphan_times = fact.join(dim_tm, "hour_of_day", "left_anti").count()

assert orphan_users == 0, f"FAIL T2: Found {orphan_users} orphan user_sk"
assert orphan_tracks == 0, f"FAIL T2: Found {orphan_tracks} orphan track_sk"
assert orphan_devices == 0, f"FAIL T2: Found {orphan_devices} orphan device_sk"
assert orphan_dates == 0, f"FAIL T2: Found {orphan_dates} orphan date_key"
assert orphan_times == 0, f"FAIL T2: Found {orphan_times} orphan hour_of_day"

inferred_cnt = dim_t.filter("is_inferred = true").count()
print(f"[PASS] T2: 0 orphan keys across all 5 dimensions. Inferred tracks resolved: {inferred_cnt}")


### Test T3: Slowly Changing Dimension (SCD Type 2) Integrity


In [ ]:
users = spark.table(GOLD['dim_user']).filter("user_sk != -1")

# Check: Exactly one current version per user
current_counts = users.filter("is_current = true").groupBy("user_id").count()
multi_current = current_counts.filter("count > 1").count()
zero_current = users.select("user_id").distinct().count() - current_counts.count()

assert multi_current == 0, f"FAIL T3: {multi_current} users have multiple current versions"
assert zero_current == 0, f"FAIL T3: {zero_current} users have zero current versions"

# Check: Temporal validity intervals
invalid_windows = users.filter("valid_from >= valid_to").count()
assert invalid_windows == 0, f"FAIL T3: Found {invalid_windows} inverted validity windows"

print(f"[PASS] T3: SCD2 integrity verified: strictly 1 active record per user, continuous time horizons.")


### Test T4: Metric Conservation


In [ ]:
fact_mins = spark.table(GOLD['fact_listening']).filter("minutes_played IS NOT NULL").selectExpr("SUM(minutes_played) as total").first()['total'] or 0.0
silver_mins = spark.table(SILVER_EVENTS).filter("ms_played IS NOT NULL").selectExpr("SUM(ms_played) / 60000.0 as total").first()['total'] or 0.0

diff = abs(fact_mins - silver_mins)
assert diff < 0.01, f"FAIL T4: Fact minutes ({fact_mins:,.2f}) differs from Silver ({silver_mins:,.2f}) by {diff}"

print(f"[PASS] T4: Metric conservation verified: Fact ({fact_mins:,.2f} min) == Silver ({silver_mins:,.2f} min).")


### Test T5: Change Data Feed (CDF) Incremental Idempotency


In [ ]:
state_df = spark.table(PIPELINE_STATE).filter(f"table_name = '{SILVER_EVENTS}'")
assert state_df.count() > 0, f"FAIL T5: No state entry for {SILVER_EVENTS} in {PIPELINE_STATE}"

cur_v = spark.sql(f"DESCRIBE HISTORY {SILVER_EVENTS} LIMIT 1").first()['version']
last_v = state_df.first()['last_version']

print(f"[PASS] T5: Fact table synchronized with Silver version: {last_v} (Current Silver Version: {cur_v}).")


### Test T6: Subscription Propensity Ground Truth Reconciliation


In [ ]:
import json

propensity_df = spark.table(GOLD['subscription_propensity_segments']).toPandas()
validation_path = os.path.join(LANDING_ROOT, "validation", "user_segments.json")

if os.path.exists(validation_path):
    with open(validation_path, "r", encoding="utf-8") as f:
        ground_truth = json.load(f)
    
    matches = 0
    total_free = 0
    for u in ground_truth:
        if not u.get("is_premium", False):
            total_free += 1
            pred = propensity_df.loc[propensity_df['user_id'] == u['user_id']]
            if not pred.empty:
                expected_flag = u.get("propensity_flag") == "High Conversion Propensity"
                actual_flag = bool(pred.iloc[0]['is_high_conversion_propensity'])
                if expected_flag == actual_flag:
                    matches += 1
                    
    match_rate = matches / max(total_free, 1)
    print(f"[PASS] T6: Propensity model match rate against ground truth: {match_rate:.1%} ({matches}/{total_free})")
    assert match_rate >= 0.80, f"FAIL T6: Match rate below 80%: {match_rate:.1%}"
else:
    print(f"[SKIP] T6: Ground truth user_segments.json not at {validation_path} (evaluates on full environment).")


### Test T7: Analytical Mart Share Sum Completeness


In [ ]:
shares = spark.sql(f"""
    SELECT year, month, ROUND(SUM(genre_market_share), 3) as total_share
    FROM {GOLD['content_performance_trends']}
    GROUP BY year, month
""").collect()

for row in shares:
    assert 0.99 <= row['total_share'] <= 1.01, f"FAIL T7: Market share for {row['year']}-{row['month']} = {row['total_share']}"

print(f"[PASS] T7: Genre market share sums to 100% (+/- 0.01) across all {len(shares)} monthly cohorts.")
